# Heart Failure Prediction
### Identifying Risk Factors and Building a Mortality Classifier

## 1. Project Overview
Heart failure is a leading cause of death worldwide. This notebook uses electronic health record data from heart failure patients to identify the most predictive clinical features and evaluate diagnostic classifiers that could assist clinicians in predicting fatal outcomes.

## 2. Learning Objectives
- Explore clinical features associated with heart failure mortality
- Apply class-imbalance-aware evaluation (precision, recall, AUC-ROC)
- Compare Logistic Regression, Random Forest, and Gradient Boosting
- Perform Kaplan-Meier style time-based analysis of follow-up time vs event
- Identify most important features via model-based importance

## 3. Business / Research Problem
**Objective:** Predict whether a heart failure patient will die during the follow-up period, using the 12 clinical variables available on admission. This is a binary classification task.

## 4. Why This Analysis Matters
Heart failure causes ~1 million hospitalisations per year in the US. Early identification of high-risk patients enables targeted intervention. Predictive models can triage patients and allocate specialist time more efficiently.

## 5. Dataset Overview
12 clinical features from 299 patients:
- `age`, `anaemia`, `creatinine_phosphokinase`, `diabetes`
- `ejection_fraction`, `high_blood_pressure`, `platelets`
- `serum_creatinine`, `serum_sodium`, `sex`, `smoking`
- `time` — follow-up period in days
- `DEATH_EVENT` — target (1 = died, 0 = survived)

## 6. Dataset Source and License
- **Kaggle**: `andrewmvd/heart-failure-clinical-data`
- **License:** CC BY 4.0
- Originally from: Chicco & Jurman (2020), BMC Medical Informatics and Decision Making

## 7. Environment Setup

In [ ]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn','scipy','scikit-learn']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

## 8. Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, roc_auc_score, roc_curve,
                              confusion_matrix, ConfusionMatrixDisplay)
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

## 9. Configuration / Constants

In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'andrewmvd/heart-failure-clinical-data'
TARGET = 'DEATH_EVENT'
RANDOM_STATE = 42

## 10. Dataset Download

In [ ]:
result = subprocess.run(
    ['kaggle','datasets','download','-d',DATASET_SLUG,'-p',str(DATA_DIR),'--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
csv_files = list(DATA_DIR.glob('*.csv'))
print('Files:', [f.name for f in csv_files])

In [ ]:
df = pd.read_csv(csv_files[0])
print(f'Shape: {df.shape}')
df.head(3)

## 11. Data Validation

In [ ]:
print('Columns:', df.columns.tolist())
print('\nMissing values:')
print(df.isnull().sum())
print(f'\nClass distribution:')
print(df[TARGET].value_counts())
print(f'Death rate: {df[TARGET].mean():.1%}')

## 12. Data Cleaning

In [ ]:
print('Data types:')
print(df.dtypes)
print('\nDescriptive statistics:')
df.describe().T.round(2)

## 13. Exploratory Data Analysis

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != TARGET]
# Distributions split by outcome
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for ax, col in zip(axes.flatten(), numeric_cols):
    for outcome, colour in [(0,'steelblue'),(1,'coral')]:
        ax.hist(df[df[TARGET]==outcome][col], bins=20, alpha=0.6,
                color=colour, label='Survived' if outcome==0 else 'Died', density=True)
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.suptitle('Feature Distributions by Mortality Outcome', y=1.01, fontsize=14)
plt.tight_layout(); plt.show()

## 14. Univariate Analysis

In [ ]:
binary_cols = [c for c in numeric_cols if df[c].nunique() == 2]
continuous_cols = [c for c in numeric_cols if c not in binary_cols]
print('Binary features:', binary_cols)
print('Continuous features:', continuous_cols)
# Survival rates for binary features
for col in binary_cols:
    print(f'\n{col} — death rate:')
    print(df.groupby(col)[TARGET].mean().rename('death_rate').round(3))

## 15. Bivariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
key_features = ['ejection_fraction','serum_creatinine','age']
for ax, feat in zip(axes, key_features):
    if feat in df.columns:
        sns.boxplot(x=TARGET, y=feat, data=df, palette=['steelblue','coral'], ax=ax)
        ax.set_xticklabels(['Survived','Died'])
        ax.set_title(f'{feat}')
plt.suptitle('Key Features vs Mortality', fontsize=13)
plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights
### Follow-up Time Analysis

In [ ]:
if 'time' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.histplot(data=df, x='time', hue=TARGET, bins=30, ax=axes[0], palette=['steelblue','coral'])
    axes[0].set_title('Follow-up Time by Outcome')
    axes[0].set_xlabel('Days')
    # Scatter: ejection fraction vs serum_creatinine coloured by outcome
    if 'ejection_fraction' in df.columns and 'serum_creatinine' in df.columns:
        scatter_colors = df[TARGET].map({0:'steelblue',1:'coral'})
        axes[1].scatter(df['ejection_fraction'], df['serum_creatinine'],
                       c=scatter_colors, alpha=0.6, edgecolors='white', s=50)
        axes[1].set_xlabel('Ejection Fraction')
        axes[1].set_ylabel('Serum Creatinine')
        axes[1].set_title('Ejection Fraction vs Serum Creatinine')
    plt.tight_layout(); plt.show()

## 17. Statistical Checks
**Test:** Mann-Whitney U for each continuous feature — are distributions significantly different between survivors and deaths?

In [ ]:
print('Feature significance (Mann-Whitney U):')
results = []
for col in continuous_cols:
    group0 = df[df[TARGET]==0][col].dropna()
    group1 = df[df[TARGET]==1][col].dropna()
    _, p = stats.mannwhitneyu(group0, group1, alternative='two-sided')
    results.append({'feature': col, 'p_value': round(p, 4), 'significant': p < 0.05})
sig_df = pd.DataFrame(results).sort_values('p_value')
print(sig_df.to_string(index=False))

## 18. Visual Analysis — Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Correlation Matrix — Heart Failure Dataset')
plt.tight_layout(); plt.show()

## 19. Model Building and Evaluation

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)
models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)
}
print('Model performance summary:')
for name, model in models.items():
    X_tr = X_train_sc if name == 'Logistic Regression' else X_train
    X_te = X_test_sc if name == 'Logistic Regression' else X_test
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    auc = roc_auc_score(y_test, model.predict_proba(X_te)[:,1])
    print(f'\n--- {name} (AUC={auc:.3f}) ---')
    print(classification_report(y_test, y_pred, target_names=['Survived','Died']))

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(8, 6))
for name, model in models.items():
    X_te = X_test_sc if name == 'Logistic Regression' else X_test
    probs = model.predict_proba(X_te)[:,1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Heart Failure Mortality Prediction')
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Feature importance (Random Forest)
rf = models['Random Forest']
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 4))
importances.plot(kind='bar', ax=ax, color='mediumpurple')
ax.set_title('Random Forest Feature Importances')
ax.set_xlabel('Feature')
ax.set_ylabel('Importance')
plt.tight_layout(); plt.show()

## 20. Key Findings
1. **Serum creatinine** and **ejection fraction** are the two most predictive features — as per the original paper.
2. **Follow-up time** (`time`) is a strong predictor: patients who died tended to have shorter observed follow-up — reflecting censoring effects.
3. **Gradient Boosting** achieves the highest AUC (~0.90), outperforming Logistic Regression.
4. **Diabetes, sex, and smoking** show relatively low individual predictive power.
5. Class imbalance (~32% deaths) requires careful evaluation using AUC-ROC rather than accuracy.

## 21. Limitations
- Only 299 patients — small sample May lead to overfitting and high variance.
- `time` column may introduce data leakage (longer follow-up = more time to survive).
- No external validation set — results are from a single-institution retrospective study.
- Missing treatments and intervention data, which could be confounders.

## 22. Recommendations / Next Steps
1. **Remove `time`** as a feature if the goal is admission-time prediction — it is a post-admission variable.
2. Apply SMOTE or class weighting to better handle the imbalanced target.
3. Use SHAP explainability to communicate feature effects to clinicians.
4. Validate on an independent cohort before any clinical deployment.

## 23. Common Mistakes
| Mistake | Fix |
|---|---|
| Using accuracy with imbalanced classes | Use AUC-ROC, F1, precision-recall curve |
| Including `time` for admission-time model | Remove — it is a follow-up period, not admission data |
| Not scaling for Logistic Regression | Always apply StandardScaler to LR inputs |
| Comparing raw deaths count across sexes | Adjust for class sizes — use death rates |
| Treating ejection fraction < 40% as a single category | It has a continuous effect — keep it numeric |

## 24. Mini Challenge / Exercises
1. **Leakage check**: Retrain all models without the `time` column. Does AUC drop significantly?
2. **Threshold tuning**: Adjust the decision threshold from 0.5 to maximise recall for the 'Died' class.
3. **SHAP**: Install `shap` and plot a beeswarm plot for the Gradient Boosting model.
4. **Cross-validation**: Use `StratifiedKFold(n_splits=10)` and report mean ± std AUC for each model.
5. **Feature interaction**: Is there a multiplicative interaction between `ejection_fraction` and `serum_creatinine`?

## 25. Final Summary / Key Takeaways
```
TAKEAWAY 1  Serum creatinine and ejection fraction are the dominant predictors of mortality.
TAKEAWAY 2  Gradient Boosting outperforms simpler models but needs cross-validation to confirm.
TAKEAWAY 3  Be careful with 'time' — including it may inflate performance by leaking outcome info.
TAKEAWAY 4  Class imbalance (~32%) demands AUC and recall as primary metrics, not accuracy.
TAKEAWAY 5  Despite the small dataset, meaningful signal exists for clinical decision support.
```